<a href="https://colab.research.google.com/github/mihaipopa15/unit-testing-junit/blob/main/ex1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

In [22]:
def synthetic_data(w, b):
    """Generate y = Xw + b + noise."""
    X = torch.arange(-3,3,0.1).reshape(60,1)
    y = X @ w + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape(-1, 1)

In [38]:
true_w = torch.tensor([1.0])
true_b = -1
features, labels = synthetic_data(true_w, true_b)

In [35]:
def data_iter(batch_size, features, labels):
    num_examples = len(features)
    indices = list(range(num_examples))
    # The examples are read at random, in no particular order
    np.random.shuffle(indices)
    for i in range(0, num_examples, batch_size):
        batch_indices = torch.tensor(
            indices[i: min(i + batch_size, num_examples)])
        yield features[batch_indices], labels[batch_indices]

In [27]:
#batches

def load_array(data_arrays, batch_size, is_train=True):
    """Construct a PyTorch data iterator."""
    dataset = torch.utils.data.TensorDataset(*data_arrays)
    return torch.utils.data.DataLoader(dataset, batch_size, shuffle=is_train)

    batch_size = 10
    data_iter = load_array((features, labels), batch_size)

In [41]:
#linear regression model
from torch import nn

net = nn.Sequential(nn.Linear(1, 1)) # asigura-te ca are un singur feature de intrare

In [30]:
#initialization of weights and bias cu 1

net[0].weight.data.normal_(1)
net[0].bias.data.fill_(1)

tensor([1.])

In [31]:
#implement loss, la ex nu va trebui , folosim  meloss
def loss_mse(y_pred, y):
  return torch.mean((y_pred-y)**2)

In [42]:
num_epochs = 25
batch_size = 10
for epoch in range(num_epochs):
    for X, y in data_iter(batch_size, features, labels):
        l = loss_mse(net(X), y)
        l.backward()
        net[0].weight.data = net[0].weight.data-0.05*net[0].weight.grad.data #w = w-learning rate*grad
        net[0].bias.data = net[0].bias.data-0.05*net[0].bias.grad.data #b = b-learning rate*grad
        net[0].weight.grad.data.zero_()
        net[0].bias.grad.data.zero_()
    l = loss_mse(net(features), labels)
    print(f'Epoch {epoch + 1}, Loss {l:f}')

Epoch 1, Loss 0.065404
Epoch 2, Loss 0.015762
Epoch 3, Loss 0.004504
Epoch 4, Loss 0.001324
Epoch 5, Loss 0.000437
Epoch 6, Loss 0.000196
Epoch 7, Loss 0.000128
Epoch 8, Loss 0.000109
Epoch 9, Loss 0.000103
Epoch 10, Loss 0.000103
Epoch 11, Loss 0.000101
Epoch 12, Loss 0.000102
Epoch 13, Loss 0.000102
Epoch 14, Loss 0.000101
Epoch 15, Loss 0.000101
Epoch 16, Loss 0.000102
Epoch 17, Loss 0.000104
Epoch 18, Loss 0.000101
Epoch 19, Loss 0.000101
Epoch 20, Loss 0.000103
Epoch 21, Loss 0.000101
Epoch 22, Loss 0.000101
Epoch 23, Loss 0.000102
Epoch 24, Loss 0.000101
Epoch 25, Loss 0.000102


In [43]:
w = net[0].weight.data
print('Error in estimating w:', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('Error in estimating b:', true_b - b)

Error in estimating w: tensor([0.0002])
Error in estimating b: tensor([-8.6546e-05])
